In [5]:
import random

class CRCTester:
    def __init__(self):
        self.crc_types = {
            'CRC-4': '10011',
            'CRC-8': '100000111',
            'CRC-16': '10001000000100001',
            'CRC-32': '100000100110000010001110110110111'
        }
        self.results = []
    
    def text_to_bits(self, text):
        return ''.join(format(ord(c), '08b') for c in text)
    
    def calculate_crc(self, data, poly):
        data = data + '0' * (len(poly) - 1)
        data = list(data)
        
        for i in range(len(data) - len(poly) + 1):
            if data[i] == '1':
                for j in range(len(poly)):
                    if data[i+j] == poly[j]:
                        data[i+j] = '0'
                    else:
                        data[i+j] = '1'
        
        return ''.join(data[-(len(poly)-1):])
    
    def run_test(self, test_num, message, crc_type, error_count):
        bits = self.text_to_bits(message)
        poly = self.crc_types[crc_type]
        crc = self.calculate_crc(bits, poly)
        encoded = bits + crc
        
        print(f"\n{'━'*60}")
        print(f"📊 TEST {test_num}: '{message}' with {crc_type}")
        print(f"{'━'*60}")
        
        print(f"📤 SENDER:")
        print(f"   Message:   '{message}'")
        print(f"   Binary:    {bits}")
        print(f"   Polynomial: {poly}")
        print(f"   CRC:       {crc}")
        print(f"   Send:      {encoded}")
        
        corrupted = list(encoded)
        error_positions = []
        
        if error_count > 0:
            error_positions = random.sample(range(len(corrupted)), error_count)
            for pos in error_positions:
                corrupted[pos] = '1' if corrupted[pos] == '0' else '0'
        
        corrupted_str = ''.join(corrupted)
        
        print(f"\n📥 RECEIVER:")
        print(f"   Received:  ", end='')
        
        for i, bit in enumerate(corrupted_str):
            if i in error_positions:
                print(f"\033[91m{bit}\033[0m", end='')
            else:
                print(bit, end='')
        
        print()
        
        if error_positions:
            print(f"   Error positions: {sorted(error_positions)}")
        
        remainder = self.calculate_crc(corrupted_str, poly)
        
        print(f"\n🔍 CHECK:")
        print(f"   Remainder: {remainder}")
        
        is_correct = all(b == '0' for b in remainder)
        
        if error_count == 0:
            print(f"   Result:    {'✅ CORRECT' if is_correct else '❌ ERROR'}")
        else:
            print(f"   Result:    {'✅ ERROR DETECTED' if not is_correct else '❌ ERROR NOT DETECTED'}")
        
        self.results.append({
            'test': test_num,
            'message': message,
            'crc_type': crc_type,
            'errors': error_count,
            'remainder': remainder,
            'detected': not is_correct if error_count > 0 else is_correct,
            'error_positions': sorted(error_positions)
        })
        
        return remainder

print("╔══════════════════════════════════════════════════════════╗")
print("║                 🚀 CRC ERROR DETECTION TEST              ║")
print("╚══════════════════════════════════════════════════════════╝")

tester = CRCTester()

tester.run_test(1, "Hi", "CRC-8", 0)
tester.run_test(2, "Hi", "CRC-8", 1)
tester.run_test(3, "Hi", "CRC-8", 2)
tester.run_test(4, "Network", "CRC-16", 0)
tester.run_test(5, "Network", "CRC-16", 2)
tester.run_test(6, "Data", "CRC-4", 1)
tester.run_test(7, "Data", "CRC-4", 3)
tester.run_test(8, "Test", "CRC-32", 0)
tester.run_test(9, "Test", "CRC-32", 1)

print(f"\n{'━'*60}")
print("📈 VISUALIZATION")
print(f"{'━'*60}")

print("\n🎯 CRC DETECTION PERFORMANCE:")
print("="*70)

detection_stats = {}
for crc_type in tester.crc_types.keys():
    relevant_tests = [r for r in tester.results if r['crc_type'] == crc_type and r['errors'] > 0]
    if relevant_tests:
        detected = sum(1 for r in relevant_tests if r['detected'])
        total = len(relevant_tests)
        detection_stats[crc_type] = (detected, total)

for crc_type, (detected, total) in detection_stats.items():
    percentage = (detected / total) * 100
    bar_length = int(percentage / 2)
    bar = '█' * bar_length + '░' * (50 - bar_length)
    print(f"{crc_type:8} [{bar}] {detected}/{total} ({percentage:.1f}%)")

print("\n" + "="*70)
print("📋 TEST RESULTS SUMMARY")
print("="*70)

print(f"{'Test':5} {'Message':10} {'CRC Type':8} {'Errors':7} {'Remainder':20} {'Result':20}")
print("-" * 70)

for result in tester.results:
    test_num = result['test']
    message = result['message']
    crc_type = result['crc_type']
    errors = result['errors']
    remainder = result['remainder']
    detected = result['detected']
    
    if errors == 0:
        result_str = "✅ OK" if detected else "❌ FAIL"
    else:
        result_str = "✅ DETECTED" if detected else "❌ MISSED"
    
    print(f"{test_num:5} {message:10} {crc_type:8} {errors:7} {remainder:20} {result_str:20}")

print("="*70)

print(f"\n🎲 Random Error Positions in Tests:")
for result in tester.results:
    if result['error_positions']:
        print(f"Test {result['test']}: Errors at {result['error_positions']}")

print(f"\n{'━'*60}")
print("💡 KEY INSIGHTS")
print(f"{'━'*60}")

print("""
1. ✅ Without errors: Remainder should be ALL ZEROS
2. ❌ With errors: Remainder should NOT be all zeros
3. 📈 Higher CRC degree = Better error detection
4. 🔍 CRC-32 detects almost all errors
5. ⚡ CRC-4 is faster but less reliable
""")


╔══════════════════════════════════════════════════════════╗
║                 🚀 CRC ERROR DETECTION TEST              ║
╚══════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 TEST 1: 'Hi' with CRC-8
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📤 SENDER:
   Message:   'Hi'
   Binary:    0100100001101001
   Polynomial: 100000111
   CRC:       11101011
   Send:      010010000110100111101011

📥 RECEIVER:
   Received:  010010000110100111101011

🔍 CHECK:
   Remainder: 00000000
   Result:    ✅ CORRECT

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 TEST 2: 'Hi' with CRC-8
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📤 SENDER:
   Message:   'Hi'
   Binary:    0100100001101001
   Polynomial: 100000111
   CRC:       11101011
   Send:      010010000110100111101011

📥 RECEIVER:
   Received:  010110000110100111101011
   Error positions: [3]

🔍 CHECK:
   Remainder: 10100010
   Resul